# **ZERO-SHOT CLASSIFICATION FOR GEMMA-3-4B**

In [2]:
# load test dataset, retrieve this from the .csv file

import pandas as pd

test_frame = pd.read_csv('fpb_test.csv')

text_col = 'text'  # text
label_col = 'label' # label
source_col = 'source' # sure why not
unique_labels = test_frame[label_col].unique().tolist()

print(f"loaded {len(test_frame)} rows.")
print(f"labels to test: {unique_labels}")

display(test_frame.head(10))

loaded 727 rows.
labels to test: ['Neutral', 'Fear', 'Optimism', 'Sadness', 'Joy']


,text,label,source,clean_text
0,the share capital of alma media corporation bu...,Neutral,FinancialPhraseBank,the share capital of alma media corporation bu...
1,the eu commission said earlier it had fined th...,Fear,FinancialPhraseBank,the eu commission said earlier it had fined th...
2,"kesko pursues a strategy of healthy , focused ...",Optimism,FinancialPhraseBank,kesko pursues a strategy of healthy focused gr...
3,down to eur5 .9 m h1 '09 3 august 2009 - finni...,Sadness,FinancialPhraseBank,down to eurNUM NUM m hNUM NUM NUM august NUM f...
4,"cencorp would focus on the development , manuf...",Neutral,FinancialPhraseBank,cencorp would focus on the development manufac...
5,"order intake , on the other hand , is expected...",Optimism,FinancialPhraseBank,order intake on the other hand is expected to ...
6,"according to kesko , the company agreed with t...",Optimism,FinancialPhraseBank,according to kesko the company agreed with the...
7,among the scandinavian companies present in st...,Neutral,FinancialPhraseBank,among the scandinavian companies present in st...
8,the terms of the financing were approved by th...,Optimism,FinancialPhraseBank,the terms of the financing were approved by th...
9,finland 's leading metals group outokumpu said...,Optimism,FinancialPhraseBank,finland s leading metals group outokumpu said ...


In [3]:
# log into huggingface

import os
import sys

google_colab = "google.colab" in sys.modules and not os.environ.get("VERTEX_PRODUCT")

if google_colab:
    # Use secret if running in Google Colab
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
else:
    # Store Hugging Face data under `/content` if running in Colab Enterprise
    if os.environ.get("VERTEX_PRODUCT") == "COLAB_ENTERPRISE":
        os.environ["HF_HOME"] = "/content/hf"
    # Authenticate with Hugging Face
    from huggingface_hub import get_token
    if get_token() is None:
        from huggingface_hub import notebook_login
        notebook_login()

In [4]:
!pip install -q transformers accelerate bitsandbytes scikit-learn

import torch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# configure model
model_id = "google/gemma-3-4b-it"

# load model and tokeniser
tokenizer = AutoTokenizer.from_pretrained(model_id)

# load in 4-bit to standard colab gpu memory
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    quantization_config=quantization_config
)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.1 MB/s eta 0:00:00


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

In [ ]:
# classify function for zero-shot classification. we don't need any correct samples, just load and go
def classify(text, candidate_labels):
    prompt = f"Classify the sentiment of the following text into one of these categories: {', '.join(candidate_labels)}.Respond with only the category name.\n\nText: {text}\nCategory:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=10, do_sample=False)
    response = tokenizer.decode(output[0], skip_special_tokens=True)
    prediction = response.split("Category:")[-1].strip().lower()

    # print(prediction)

    # Find the best match in candidate labels
    for label in candidate_labels:
        if label.lower() in prediction:
            return label
    return "unknown" # did something go wrong?

# time to evaluate!
y_true = test_frame[label_col].tolist() # all the correct labels
y_pred = [classify(text, unique_labels) for text in test_frame[text_col]] # all the predicted labels


In [10]:
# basic metrics
accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, average=None, labels=unique_labels, zero_division=0)

# Create a DataFrame for per-class F1 scores
per_class_f1_df = pd.DataFrame({
    'Category': unique_labels,
    'Precision': precision,
    'Recall': recall,
    'F1 Score': f1,
    'Support': support
})

print("Per-class F1 Scores:")
display(per_class_f1_df)

# For overall metrics, we can still calculate them (e.g., weighted average)
precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', labels=unique_labels, zero_division=0)
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', labels=unique_labels, zero_division=0)

# export metrics (overall weighted metrics)
results_dict = {
    'Metric': ['Accuracy', 'Precision (Weighted)', 'Recall (Weighted)', 'F1 Score (Weighted)', 'Precision (Macro)', 'Recall (Macro)', 'F1 Score (Macro)'] and ['Accuracy', 'Precision (Weighted)', 'Recall (Weighted)', 'F1 Score (Weighted)', 'Precision (Macro)', 'Recall (Macro)', 'F1 Score (Macro)'],
    'Value': [accuracy, precision_weighted, recall_weighted, f1_weighted, precision_macro, recall_macro, f1_macro]
}

# create dataframe
results_df = pd.DataFrame(results_dict)
display(results_df)

dashboard_dict = {
    'model': 'Gemma 3 4B',
    'strategy': 'zero-shot',
    'accuracy': accuracy,
    'f1_macro': f1_macro,
    'f1_weighted': f1_weighted,
    'f1_fear': per_class_f1_df.at[1, 'F1 Score'],
    'f1_joy': per_class_f1_df.at[4, 'F1 Score'],
    'f1_neutral': per_class_f1_df.at[0, 'F1 Score'],
    'f1_optimism': per_class_f1_df.at[2, 'F1 Score'],
    'f1_sadness': per_class_f1_df.at[3, 'F1 Score']
}

dashboard_df = pd.DataFrame(dashboard_dict, index=[0])

# export to csv
dashboard_df.to_csv('zero_shot_gemma_dashboard.csv', index=False)

print("Results saved to 'zero_shot_gemma_dashboard.csv'")
display(dashboard_df)

Per-class F1 Scores:


,Category,Precision,Recall,F1 Score,Support
0,Neutral,0.686106,0.925926,0.788177,432
1,Fear,0.071429,0.142857,0.095238,7
2,Optimism,0.756098,0.158163,0.261603,196
3,Sadness,0.921569,0.566265,0.701493,83
4,Joy,0.000000,0.000000,0.000000,9


,Metric,Value
0,Accuracy,0.658872
1,Precision (Weighted),0.717446
2,Recall (Weighted),0.658872
3,F1 Score (Weighted),0.619886
4,Precision (Macro),0.487040
5,Recall (Macro),0.358642
6,F1 Score (Macro),0.369302


Results saved to 'zero_shot_gemma_dashboard.csv'


,model,strategy,accuracy,f1_macro,f1_weighted,f1_fear,f1_joy,f1_neutral,f1_optimism,f1_sadness
0,Gemma 3 4B,zero-shot,0.658872,0.369302,0.619886,0.095238,0.0,0.788177,0.261603,0.701493
